# Text-target EEG alignment

This standalone report documents the completed text-target alignment experiment. Qwen3.5 generates image captions, `t5_base` embeds those captions, and the EEG encoder is trained to retrieve the matching text embedding. The report reads saved artifacts only; it does not launch training or require a GPU.

## Maintained pipeline and protocol

- Caption generator: `Qwen/Qwen3.5-0.8B`
- Alignment target: frozen `t5_base` text embeddings (768 dimensions)
- Trainable component: EEG encoder only
- Objective: symmetric InfoNCE
- EEG augmentation: disabled
- MSE alignment loss: disabled
- Dataset: Things-EEG2 subject 8
- Batch size: 32; DataLoader workers: 4
- Checkpoint criterion: maximum validation alignment top-1 accuracy

The maintained configuration is [`train_eeg_align_text.yaml`](../src/brain_image/configs/train_eeg_align_text.yaml). Training is implemented by [`train_eeg.py`](../scripts/training/train_eeg.py), and evaluation is separate in [`test_eeg.py`](../scripts/evaluation/test_eeg.py).

In [1]:
from pathlib import Path
import json
import pandas as pd
import yaml
from IPython.display import display

repo = Path('..').resolve()
caption_path = repo / 'data/things-eeg2/captions/qwen3.5-0.8b.jsonl'
cache_dir = repo / 'tensorcache/qwen3.5-0.8b/t5_base'
stats_dir = repo / 'statistics/qwen3.5-0.8b'
run_dir = repo / 'experiments/eeg_alignment_text/20260904_002351/train_eeg-20260904_002425/version_0'
eval_dir = run_dir / 'test'
assert caption_path.exists() and cache_dir.exists() and stats_dir.exists() and eval_dir.exists()
print('Artifacts and evaluation directory found.')

Artifacts and evaluation directory found.


In [2]:
caption_records = [json.loads(line) for line in caption_path.open()]
caption_paths = [record['path'] for record in caption_records]
caption_summary = pd.DataFrame({
    'artifact': ['captions', 'train captions', 'test captions', 'T5 cache files'],
    'count': [len(caption_records), sum('training_images' in p for p in caption_paths),
              sum('test_images' in p for p in caption_paths), len(list(cache_dir.rglob('*.pt')))],
})
display(caption_summary)
assert len(caption_records) == 16740
assert len(set(caption_paths)) == len(caption_paths)
assert all(record['caption'].strip() for record in caption_records)
print('Caption coverage and uniqueness checks passed.')

,artifact,count
0,captions,16740
1,train captions,16540
2,test captions,200
3,T5 cache files,16740


Caption coverage and uniqueness checks passed.


In [3]:
metrics = pd.read_csv(eval_dir / 'test_metrics.csv')
display(metrics)
expected = {'eval/test/align/brain_acc', 'eval/test/align/image_acc', 'eval/test/align/sim'}
assert set(metrics['metric']) == expected
metric_values = metrics.set_index('metric')['value']
assert abs(metric_values['eval/test/align/image_acc'] - 0.1700000018) < 1e-6
print(f"Primary image_acc: {metric_values['eval/test/align/image_acc']:.3f}")
print(f"Reverse brain_acc: {metric_values['eval/test/align/brain_acc']:.3f}")

,metric,value
0,eval/test/align/brain_acc,0.21500
1,eval/test/align/image_acc,0.17000
2,eval/test/align/sim,0.21608


Primary image_acc: 0.170
Reverse brain_acc: 0.215


In [4]:
config = yaml.safe_load((eval_dir / 'evaluation_config.yaml').read_text())
checkpoint_files = sorted((run_dir / 'checkpoints').glob('*.ckpt'))
pd.DataFrame({
    'item': ['selected checkpoint', 'checkpoint criterion', 'training TensorBoard', 'evaluation metrics'],
    'value': [checkpoint_files[0].name if checkpoint_files else 'not found',
              'max val/align/top1', 'events.out.tfevents.*', 'test/test_metrics.csv'],
}).style.hide(axis='index')

item,value
selected checkpoint,epoch_0029-val_align_top1_0.5250.ckpt
checkpoint criterion,max val/align/top1
training TensorBoard,events.out.tfevents.*
evaluation metrics,test/test_metrics.csv


## Result and interpretation

The full run trained until early stopping at epoch 129. The best saved checkpoint was `epoch_0029-val_align_top1_0.5250.ckpt`. On the 200-image test split, the evaluator reported `image_acc=0.170`, `brain_acc=0.215`, and cosine similarity `0.2161`.

`image_acc` is the primary EEG-to-text retrieval metric for this experiment; `brain_acc` is retained as the reverse retrieval direction. Evaluation writes CSV/YAML artifacts separately from the training TensorBoard logs.